# Setup

In [1]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torchvision.transforms as T
import torchvision.models as models

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Step 2: Load Labels and Images

In [2]:
# Paths
base_dir = Path.cwd()
img_dir = base_dir / "labeled_example_data"
labels_path = img_dir / "labels.json"

# Load labels
with open(labels_path) as f:
    labels = json.load(f)

# Filter valid samples
samples = [e for e in labels if e["is_present"] == 1.0]

In [3]:
print(samples[1])  # Example entry

{'filename': 'rim_001.png', 'is_present': 1.0, 'center': [0.7964510178183489, 0.4742496374881629]}


Feature Extractor (ResNet18)

In [4]:
# Pretrained ResNet18 (remove final layer)
resnet = models.resnet18(pretrained=True)
resnet.fc = torch.nn.Identity()
resnet.eval()

# Image transform
transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Extract features
features = []
targets = []

for entry in samples:
    fname = entry["filename"]
    img_path = img_dir / fname
    if not img_path.exists():
        continue
    img = Image.open(img_path).convert("RGB")
    x_norm, y_norm = entry["center"]

    # Transform and extract features
    with torch.no_grad():
        tensor = transform(img).unsqueeze(0)  # shape: (1, 3, 128, 128)
        feat = resnet(tensor).squeeze().numpy()  # shape: (512,)
        features.append(feat)
        targets.append([x_norm, y_norm])

/Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


# Train XGBoost Regressors

In [5]:
# Convert to arrays
X = np.array(features)
y = np.array(targets)

# Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train two regressors: one for x, one for y
model_x = xgb.XGBRegressor(n_estimators=100, max_depth=6)
model_y = xgb.XGBRegressor(n_estimators=100, max_depth=6)

model_x.fit(X_train, y_train[:, 0])
model_y.fit(X_train, y_train[:, 1])

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


# Evaluate

In [6]:
# Predict
pred_x = model_x.predict(X_val)
pred_y = model_y.predict(X_val)

# Compute pixel error
H, W = 128, 128  # image size
gt_px = y_val[:, 0] * W
gt_py = y_val[:, 1] * H
pr_px = pred_x * W
pr_py = pred_y * H

errors = np.sqrt((gt_px - pr_px)**2 + (gt_py - pr_py)**2)
print(f"✅ Mean pixel error: {np.mean(errors):.2f}")
print(f"📏 Median pixel error: {np.median(errors):.2f}")

✅ Mean pixel error: 18.81
📏 Median pixel error: 16.09


# Visulaize the prediction

In [7]:
import cv2
import numpy as np
from pathlib import Path

# Folder to save overlays
overlay_dir = Path(base_dir) / "overlays_all"
overlay_dir.mkdir(exist_ok=True)

for i in range(len(fn_val)):
    fname = fn_val[i]
    img_path = img_dir / fname
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not load {fname}")
        continue

    H, W = img.shape[:2]

    # Ground truth (normalized → pixel)
    gt_x = int(round(y_val[i][0] * W))
    gt_y = int(round(y_val[i][1] * H))
    cv2.circle(img, (gt_x, gt_y), 6, (0, 0, 255), -1)  # red filled
    cv2.drawMarker(img, (gt_x, gt_y), (0, 0, 255),
                   cv2.MARKER_CROSS, 18, 2)

    # Prediction (normalized → pixel)
    pr_x = int(round(pred_x[i] * W))
    pr_y = int(round(pred_y[i] * H))
    cv2.circle(img, (pr_x, pr_y), 6, (0, 255, 0), 2)   # green ring
    cv2.drawMarker(img, (pr_x, pr_y), (0, 255, 0),
                   cv2.MARKER_TILTED_CROSS, 18, 2)

    # Optional: show error on image
    err_px = np.hypot(gt_x - pr_x, gt_y - pr_y)
    cv2.putText(img, f"err={err_px:.2f}px",
                (8, 22), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (255, 255, 0), 2, cv2.LINE_AA)

    # Save overlay
    out_path = overlay_dir / f"overlay_{fname}"
    cv2.imwrite(str(out_path), img)

print(f"✅ Saved {len(fn_val)} overlays to: {overlay_dir}")

NameError: name 'fn_val' is not defined